# Cartograma de Dorling para Analise de Criminalidade em Belem (PA)
### Pipeline de Processamento Geoespacial e Interpolacao Geometrica em R

Este caderno executa o fluxo computacional em R estruturado a partir das metodologias de **Karim Douïeb** e **David Zumbach (`votemapswitzerland`)**:
1. **Configuracao rapida de dependencias:** binarios Linux pre-compilados via Posit PPM (instalacao em segundos).
2. **Extracao dinamica no SIAC/SEGUP:** janela deslizante automatica dos ultimos 365 dias (D-365 ate Hoje).
3. **Tratamento metrico (EPSG:31982):** unificacao dos 72 bairros limpos de Belem com engine GEOS.
4. **Simulacao fisica de particulas:** resolucao de sobreposicoes via `{particles}` e `{tidygraph}`.
5. **Visualizacao 1:** Morphing continuo in-situ (Dorling Cartogram).
6. **Visualizacao 2:** Narrativa em 4 atos com ordenamento em grade e enxames comparativos por **Severidade Estatistica** (Acima vs Abaixo da Media).

## 1. Instalacao Otimizada de Dependencias (Binarios Posit PPM)

In [ ]:
# 1. Sincronizacao automatica do repositorio no ambiente Colab
message(">> Sincronizando scripts e modulos mais recentes do GitHub...")
system("git clone --depth 1 https://github.com/samuel-c-santos/belem-crime-dorling-cartogram.git /tmp/repo && cp -r /tmp/repo/* . && rm -rf /tmp/repo")
message(">> Estrutura sincronizada: pastas R/, scripts/ e data/ estao disponiveis.")

# 2. Configuracao do repositorio Posit Public Package Manager (PPM) para Ubuntu 22.04 Jammy
options(repos = c(CRAN = "https://packagemanager.posit.co/cran/__linux__/jammy/latest"))

# Dependencias do sistema e do script de extracao Python
system("apt-get update -qq && apt-get install -y -qq libudunits2-dev libgdal-dev libgeos-dev libproj-dev cargo python3-pip", intern = TRUE)
system("pip3 install -q requests pandas openpyxl", intern = TRUE)

# Pacotes R analiticos e graficos
pacotes <- c(
  "sf", "tidyverse", "igraph", "tidygraph", "particles", 
  "transformr", "gganimate", "gifski", "viridis", "scales", "stringi", "base64enc",
  "sysfonts", "showtext"
)

novos_pacotes <- pacotes[!(pacotes %in% installed.packages()[, "Package"])]
if (length(novos_pacotes) > 0) {
  install.packages(novos_pacotes, quiet = TRUE)
}

# Configuracao da tipografia tecnica via Google Fonts (Barlow, Inter, IBM Plex Sans)
suppressPackageStartupMessages(library(showtext))
font_add_google("Barlow", "barlow")
font_add_google("Inter", "inter")
font_add_google("IBM Plex Sans", "plex")
showtext_auto()

# Funcao utilitaria para exibir GIFs diretamente no display do Colab
exibir_gif_colab <- function(caminho_gif) {
  if (file.exists(caminho_gif)) {
    img_b64 <- base64enc::base64encode(caminho_gif)
    IRdisplay::display_html(sprintf('<img src="data:image/gif;base64,%s" style="display: block; margin: 0 auto; max-width: 100%%;">', img_b64))
  } else {
    warning("Arquivo nao encontrado: ", caminho_gif)
  }
}

message(">> Ambiente e dependencias configurados com sucesso!")

## 2. Extracao Dinamica de Dados no SIAC / SEGUP-PA (Janela Deslizante)

In [ ]:
# Janela deslizante automatica: data atual menos 365 dias (D-365 ate Hoje)
data_fim    <- as.character(Sys.Date())
data_inicio <- as.character(Sys.Date() - 365)

message(sprintf(">> Disparando requisicao HTTP ao SIAC/CODEC para janela deslizante de 1 ano (%s ate %s)...", data_inicio, data_fim))
cmd <- sprintf("python3 scripts/01_extrair_siac.py %s %s data", data_inicio, data_fim)
system(cmd)
message(">> Extracao concluida e arquivos tabulares gerados em data/processed/.")

## 3. Carga dos Modulos R e Tratamento Metrico dos Bairros

In [ ]:
source("R/01_geometrias_bairros.R")
source("R/02_dorling_fisica.R")
source("R/03_animacao_morphing.R")
source("R/04_animacao_narrativa_estagios.R")

bairros_dados <- preparar_geometrias_bairros(
  geojson_path = "data/raw/bairros_belem.geojson",
  crimes_csv_path = "data/processed/crimes_bairros_agregado.csv",
  crs_metrico = 31982
)

# Exibir os 5 bairros com maior volume de ocorrencias registradas
bairros_dados %>%
  st_drop_geometry() %>%
  select(BAIRRO_NORM, TOTAL_CRIMES, ROUBO, FURTO, HOMICIDIO, area_km2) %>%
  arrange(desc(TOTAL_CRIMES)) %>%
  head(5)

## 4. Simulacao Fisica de Particulas (Dorling In-Situ Layout)

In [ ]:
bairros_dorling <- calcular_dorling_layout(
  bairros_sf = bairros_dados,
  coluna_metrica = "TOTAL_CRIMES",
  fator_escala_k = 40,
  padding = 35,
  iteracoes = 180
)

message(">> Simulacao de forcas concluida e posicoes in-situ estabilizadas.")

## 5. Visualizacao 1: Morphing In-Situ (Dorling Cartogram)

In [ ]:
output_gif_1 <- gerar_animacao_cartograma(
  bairros_dorling_sf = bairros_dorling,
  coluna_valor = "TOTAL_CRIMES",
  output_file = "outputs/animacao_dorling_seguranca_belem.gif",
  cor_fundo = "#ffffff",
  n_frames = 80,
  fps = 16,
  largura = 800,
  altura = 800
)

# Exibicao no Colab
exibir_gif_colab(output_gif_1)

## 6. Visualizacao 2: Narrativa em 4 Atos com Severidade Estatistica

In [ ]:
output_gif_2 <- gerar_animacao_votemap2(
  bairros_sf = bairros_dados,
  coluna_valor = "TOTAL_CRIMES",
  output_file = "outputs/animacao_narrativa_estagios_belem.gif",
  cor_fundo = "#ffffff",
  n_frames = 110,
  fps = 16,
  largura = 800,
  altura = 800
)

# Exibicao no Colab
exibir_gif_colab(output_gif_2)